# Part 2 Procedure Conformance

## tl;dr

`mark 1 (part 2)`'s delivery contract (`AGENTS.md`, `06_WORKING_DIRECTORY_AND_DELIVERY_RULES.md`,
`07_SUCCESS_CRITERIA_PARAMETERS_AND_OUTPUTS.md`) requires every analytical phase to save
`configuration.json`, `provenance.json`, `expected_vs_actual.csv` and a machine-readable
`gate_result.json` carrying the standard schema (`result_level`, `selected_configuration`,
`selected_metrics`, `targets`, `target_passes`, `all_mandatory_targets_passed`, `decision`,
`next_step`, `manifest_sha256`, `input_artifact_hashes`, `test_images_accessed`).

This notebook makes the Evaluation suite conform to that procedure **without re-running any research
phase and without writing into `mark 1/`**. It is a read-only derivation layer: it reads the existing
executed gates + metrics under `Evaluation/output/`, maps each phase's original status onto the Part 2
result-level vocabulary, and writes the four standard contract files for every phase into
`Evaluation/output/12_procedure_conformance/data/<phase>/`. It then verifies the machine-readable gate
schema and produces a conformance gate + summary + dashboard.

## Outputs written

```
Evaluation/output/12_procedure_conformance/
├── data/<phase>/configuration.json     … explicit parameters (dataset, preproc, model, inference, metrics)
├── data/<phase>/provenance.json        … sources, hashes, counts, software, test state
├── data/<phase>/expected_vs_actual.csv … metric, value, target, direction, pass per gate row
├── data/<phase>/gate_result.json       … full Part 2 gate schema (merges the original phase gate)
├── data/procedure_conformance_gate.json
├── data/procedure_conformance_summary.csv
└── figures/procedure_conformance_dashboard.png
```

## Rules

- Test split stays locked: every gate declares `test_images_accessed: false` and the manifest hash is
  asserted.
- Nothing under `mark 1/`, `mark 1 (part 2)/` or the legacy folders is created or modified.
- Re-running is idempotent: files are regenerated from the current gates.


In [1]:
import hashlib, json, platform, sys
from datetime import datetime, timezone
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

EVAL_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation")
OUTPUT_ROOT = EVAL_ROOT / "output"
CONF_DIR = OUTPUT_ROOT / "12_procedure_conformance"
CONF_DATA = CONF_DIR / "data"
CONF_FIGS = CONF_DIR / "figures"
CONF_DATA.mkdir(parents=True, exist_ok=True)
CONF_FIGS.mkdir(parents=True, exist_ok=True)

DATASET_ROOT = Path(r"D:\DATA SCIENCE AND ANALYTICS\Dataset\Liver\02_staging\build_corrected_20260713_214847_v2")
MANIFEST_PATH = DATASET_ROOT / "manifests" / "slice_manifest.csv"
EXPECTED_MANIFEST_SHA256 = "575a6fc391d63dc9bbbbb3317efe4d0f65edd57457ae63c1bfc6c2c615861889"
SOURCE_CHECKPOINT_SHA256 = "9c4160bbd68891f9dc4e5f04ceca4391f38c5869b3f81c72b95d4639e0572223"

# Shared parameters frozen across the Evaluation suite (identical to the notebook setup cell).
BROAD_WINDOW = (-160.0, 240.0)
LIVER_WINDOW = (0.0, 200.0)
ROI_SIZE = 256
SEED = 42
CONTINUATION_TARGETS = {
    "mean_patient_dice": (0.3329, "higher"),
    "volume_104_dice": (0.05, "higher"),
    "volume_116_dice": (0.01, "higher"),
    "q1_detected_pct": (35.0, "higher"),
    "positive_predicted_empty_pct": (35.0, "lower"),
    "empty_slice_false_positive_pct": (20.0, "lower"),
}
FINAL_TARGETS = {
    "mean_patient_dice": (0.406915, "higher"),
    "volume_104_dice": (0.50, "higher"),
    "volume_116_dice": (0.05, "higher"),
    "q1_detected_pct": (45.0, "higher"),
    "positive_predicted_empty_pct": (20.0, "lower"),
    "empty_slice_false_positive_pct": (15.0, "lower"),
}

SOFTWARE = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": __import__("torch").__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}


def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


print("setup ok")


setup ok


### Phase metadata

Each phase maps its original gate status onto a Part 2 `result_level`, declares the decision vocabulary,
the next step, and the gate rows used to build `expected_vs_actual.csv`.


In [2]:
FOLDER_BY_PHASE = {
    "mark_1": "01_mark_1", "mark_2": "02_mark_2", "mark_3": "03_mark_3",
    "mark_4": "04_mark_4", "mark_4b": "05_mark_4b", "mark_4c": "06_mark_4c",
    "mark_4d": "07_mark_4d", "mark_4e": "08_mark_4e", "consolidated": "09_consolidated",
}
GATE_FILE_BY_PHASE = {
    "mark_1": "mark_1_gate_result.json", "mark_2": "mark_2_gate_result.json",
    "mark_3": "mark_3_gate_result.json", "mark_4": "mark_4_gate_result.json",
    "mark_4b": "mark_4b_gate_result.json", "mark_4c": "mark_4c_gate_result.json",
    "mark_4d": "mark_4d_gate_result.json", "mark_4e": "mark_4e_gate_result.json",
    "consolidated": "reproduction_verification.json",
}

PHASE_META = {
    "mark_1": dict(
        result_level="DIAGNOSTIC_COMPLETE",
        description="Mark 1 probability-contrast localization diagnostic; calibration gate FAILED (mislocalized/absent signal).",
        decision="PREDICTED_LIVER_ROI_OR_CAPACITY_EXPERIMENT",
        next_step="mark_2_roi_multiwindow_feasibility",
        target_set="continuation",
        metrics_source="best_observed_configuration_for_diagnosis",
    ),
    "mark_2": dict(
        result_level="DIAGNOSTIC_COMPLETE",
        description="Mark 2 predicted-liver ROI feasibility; 100% tumour containment, median crop 42.7%.",
        decision="PROCEED_TO_GATED_TWO_STAGE_MULTIWINDOW_OVERFIT",
        next_step="mark_3_two_stage_multiwindow_overfit",
        target_set="custom_roi",
        metrics_source="selected_roi_configuration",
    ),
    "mark_3": dict(
        result_level="DIAGNOSTIC_COMPLETE",
        description="Mark 3 capacity overfit gate; hard micro-Dice 0.9006 (17 ep, 1ch).",
        decision="PROCEED_TO_3_TO_5_EPOCH_TWO_STAGE_VALIDATION_SMOKE",
        next_step="mark_4_two_stage_validation_smoke",
        target_set="custom_overfit",
        metrics_source="selected_configuration",
    ),
    "mark_4": dict(
        result_level="FAILED_GATE",
        description="Mark 4 5-epoch validation smoke; 5/6 continuation targets, positive predicted-empty 36.85% > 35%.",
        decision="STOP_AND_DIAGNOSE_TWO_STAGE_SMOKE_FAILURE",
        next_step="mark_4b_threshold_diagnostics",
        target_set="continuation",
        metrics_source="best_metrics",
    ),
    "mark_4b": dict(
        result_level="PARTIAL_PASS",
        description="Mark 4B threshold sweep; threshold tuning cannot fix recall (5/6).",
        decision="PROBABILITIES_REMAIN_INSUFFICIENT_REVISE_INPUT_OR_OBJECTIVE",
        next_step="mark_4c_two_channel_or_recall_ablation",
        target_set="continuation",
        metrics_source="selected_metrics",
    ),
    "mark_4c": dict(
        result_level="FAILED_GATE",
        description="Mark 4C ablation; two-channel collapsed and recall-loss killed V116, no arm passed all targets.",
        decision="REVISE_SAMPLING_OR_ARCHITECTURE_BEFORE_MORE_TRAINING",
        next_step="mark_4d_metric_reconciliation_v116_diagnostic",
        target_set="continuation",
        metrics_source="control_arm",
    ),
    "mark_4d": dict(
        result_level="PARTIAL_PASS",
        description="Mark 4D reconciliation; V116 is a localization/recognition failure (5/6).",
        decision="V116_LOCALIZATION_FAILURE_RUN_SMALL_LESION_SAMPLING_ABLATION",
        next_step="mark_4e_checkpoint_fusion_validation",
        target_set="continuation",
        metrics_source="selected_metrics",
    ),
    "mark_4e": dict(
        result_level="TEMPORARY_CONTINUATION_PASS",
        description="Mark 4E checkpoint fusion; pixelwise maximum @ 0.70 passes all 6 continuation targets.",
        decision="FREEZE_FUSION_POLICY_AND_THRESHOLD",
        next_step="mark_4f_fusion_freeze_and_bounded_confirmation",
        target_set="continuation",
        metrics_source="selected_metrics",
    ),
    "consolidated": dict(
        result_level="DIAGNOSTIC_COMPLETE",
        description="Consolidated reproduction verification; 56/56 comparisons, worst abs diff 0.0.",
        decision="REPRODUCTION_VERIFIED",
        next_step="part2_procedure_conformance",
        target_set="custom_reproduction",
        metrics_source=None,
    ),
}


In [3]:
def load_gate(phase):
    folder = OUTPUT_ROOT / FOLDER_BY_PHASE[phase] / "data"
    gate_file = GATE_FILE_BY_PHASE[phase]
    path = folder / gate_file
    if not path.is_file():
        raise FileNotFoundError(f"Missing phase gate: {path} — run notebook for {phase} first.")
    return json.loads(path.read_text(encoding="utf-8")), path


def build_expected_vs_actual(phase, gate):
    meta = PHASE_META[phase]
    rows = []
    if meta["target_set"] == "continuation":
        if meta["metrics_source"] == "control_arm":
            src = {}
            for arm in gate.get("arms") or []:
                if arm.get("arm") == "control":
                    src = {k: v for k, v in arm.items() if k not in ("arm", "epoch", "validation_loss")}
        else:
            src = gate.get(meta["metrics_source"]) or gate.get("selected_metrics") or {}
        for metric, (target, direction) in CONTINUATION_TARGETS.items():
            value = src.get(metric, np.nan)
            if isinstance(value, (int, float)) and not isinstance(value, bool):
                passed = value >= target if direction == "higher" else value <= target
            else:
                passed = np.nan
            rows.append({"phase": phase, "metric": metric, "value": value, "target": target,
                         "direction": direction, "target_set": "continuation", "passed": passed})
    elif meta["target_set"] == "custom_roi":
        cfg = gate.get("selected_roi_configuration") or {}
        def row(name, target, direction="higher"):
            v = cfg.get(name, np.nan)
            p = (v >= target) if direction == "higher" else (v <= target)
            rows.append({"phase": phase, "metric": name, "value": v, "target": target,
                         "direction": direction, "target_set": "roi", "passed": p})
        row("hard_containment_gate_passed", True)
        row("efficient_roi_gate_passed", True)
        row("volume_104_tumor_containment", 1.0)
        row("volume_116_tumor_containment", 1.0)
        row("empty_patient_rois", 0, "lower")
    elif meta["target_set"] == "custom_overfit":
        cfg = gate.get("selected_configuration") or {}
        def row(name, target, direction="higher"):
            v = cfg.get(name, np.nan)
            p = (v >= target) if direction == "higher" else (v <= target)
            rows.append({"phase": phase, "metric": name, "value": v, "target": target,
                         "direction": direction, "target_set": "overfit", "passed": p})
        row("best_hard_micro_dice", 0.85)
        row("positive_predicted_empty_pct", 5.0, "lower")
        for flag in ("training_roi_gate_passed", "geometry_gate_passed"):
            rows.append({"phase": phase, "metric": flag, "value": gate.get(flag, np.nan),
                         "target": True, "direction": "higher", "target_set": "overfit",
                         "passed": gate.get(flag, np.nan) is True})
    elif meta["target_set"] == "custom_reproduction":
        ver = gate
        comparisons_passed = ver.get("comparisons_passed")
        rows.append({"phase": phase, "metric": "comparisons_passed", "value": comparisons_passed,
                     "target": 56, "direction": "higher", "target_set": "reproduction",
                     "passed": bool(comparisons_passed is not None and comparisons_passed >= 56)})
        rows.append({"phase": phase, "metric": "worst_abs_diff", "value": ver.get("worst_abs_diff", np.nan),
                     "target": 1e-4, "direction": "lower", "target_set": "reproduction",
                     "passed": bool(ver.get("worst_abs_diff") is not None and ver.get("worst_abs_diff") <= 1e-4)})
        rows.append({"phase": phase, "metric": "all_passed", "value": ver.get("all_passed", np.nan),
                     "target": True, "direction": "higher", "target_set": "reproduction",
                     "passed": ver.get("all_passed") is True})
    return pd.DataFrame(rows)


In [4]:
# Build all four Part 2 contract files per phase from the executed gates.
SUMMARY = []
for phase in sorted(FOLDER_BY_PHASE):
    gate, gate_path = load_gate(phase)
    meta = PHASE_META[phase]
    phase_dir = CONF_DATA / phase
    phase_dir.mkdir(parents=True, exist_ok=True)

    eva = build_expected_vs_actual(phase, gate)
    eva.to_csv(phase_dir / "expected_vs_actual.csv", index=False, float_format="%.6f")

    if eva.empty:
        all_pass, pass_count, total_count = None, 0, 0
    else:
        numeric = eva.dropna(subset=["passed"])
        pass_count = int(numeric["passed"].astype(bool).sum())
        total_count = len(eva)
        all_pass = bool(pass_count == total_count and total_count > 0)

    # ---- selected metrics (six-metric suite where applicable) ----
    selected_metrics = {}
    src = gate.get(meta["metrics_source"]) if meta["metrics_source"] else None
    if isinstance(src, dict):
        for k, v in src.items():
            if isinstance(v, (int, float)) and not isinstance(v, bool):
                selected_metrics[k] = v
    elif meta["metrics_source"] == "control_arm":
        arms = gate.get("arms") or []
        for arm in arms:
            if arm.get("arm") == "control":
                for k, v in arm.items():
                    if k not in ("arm", "epoch", "validation_loss") and isinstance(v, (int, float)):
                        selected_metrics[k] = v

    targets = {m: t for m, (t, _d) in CONTINUATION_TARGETS.items()}
    target_passes = {}
    if meta["target_set"] == "continuation":
        for row in eva.itertuples(index=False):
            if row.metric in targets:
                target_passes[row.metric] = bool(row.passed) if row.passed == row.passed else None

    configuration = {
        "phase": phase,
        "description": meta["description"],
        "dataset_build_id": "build_corrected_20260713_214847_v2",
        "dataset_root": str(DATASET_ROOT),
        "manifest_path": str(MANIFEST_PATH),
        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
        "random_seed": SEED,
        "windows": {"broad_hu": list(BROAD_WINDOW), "liver_hu": list(LIVER_WINDOW)},
        "roi": {"resize": ROI_SIZE, "liver_threshold": 0.5, "component_rule": "largest_3d", "padding": 16},
        "source_checkpoint_sha256": SOURCE_CHECKPOINT_SHA256,
        "source_checkpoint_path": str(EVAL_ROOT.parent / "Practice" / "multitask_liver_tumor_outputs" / "multitask_best.pth"),
        "test_images_accessed": False,
        "analysis_mode": "REPRODUCTION_REUSE",
        "original_gate": gate.get("status"),
    }
    write_json(phase_dir / "configuration.json", configuration)

    provenance = {
        "phase": phase,
        "generated_by": "12_part2_procedure_conformance",
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "dataset_build_id": configuration["dataset_build_id"],
        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
        "source_checkpoint_sha256": SOURCE_CHECKPOINT_SHA256,
        "input_artifact_hashes": {
            "phase_gate_sha256": sha256_file(gate_path),
            "phase_gate_name": gate_path.name,
        },
        "test_images_accessed": False,
        "software": SOFTWARE,
    }
    write_json(phase_dir / "provenance.json", provenance)

    gate_result = {
        "status": gate.get("status") or phase,
        "result_level": meta["result_level"],
        "phase": phase,
        "description": meta["description"],
        "selected_configuration": {k: v for k, v in (gate.get("selected_configuration") or {}).items()
                                   if not isinstance(v, (list, dict))} if gate.get("selected_configuration") else {},
        "selected_metrics": selected_metrics,
        "targets": targets,
        "target_passes": target_passes,
        "all_mandatory_targets_passed": all_pass,
        "targets_passed": pass_count,
        "targets_total": total_count,
        "decision": meta["decision"],
        "next_step": meta["next_step"],
        "manifest_sha256": EXPECTED_MANIFEST_SHA256,
        "input_artifact_hashes": {"phase_gate_sha256": sha256_file(gate_path)},
        "test_images_accessed": False,
        "original_gate_json": {k: v for k, v in gate.items() if k not in ("targets", "target_passes")},
    }
    write_json(phase_dir / "gate_result.json", gate_result)

    SUMMARY.append({"phase": phase, "result_level": meta["result_level"],
                    "target_set": meta["target_set"], "rows": total_count,
                    "passed": pass_count, "all_mandatory_targets_passed": all_pass,
                    "test_images_accessed": False, "gate_file": gate_path.name})

summary_df = pd.DataFrame(SUMMARY)
summary_df.to_csv(CONF_DATA / "procedure_conformance_summary.csv", index=False, float_format="%.6f")
print(summary_df.to_string(index=False))


       phase                result_level          target_set  rows  passed  all_mandatory_targets_passed  test_images_accessed                      gate_file
consolidated         DIAGNOSTIC_COMPLETE custom_reproduction     3       3                          True                 False reproduction_verification.json
      mark_1         DIAGNOSTIC_COMPLETE        continuation     6       1                         False                 False        mark_1_gate_result.json
      mark_2         DIAGNOSTIC_COMPLETE          custom_roi     5       5                          True                 False        mark_2_gate_result.json
      mark_3         DIAGNOSTIC_COMPLETE      custom_overfit     4       4                          True                 False        mark_3_gate_result.json
      mark_4                 FAILED_GATE        continuation     6       5                         False                 False        mark_4_gate_result.json
     mark_4b                PARTIAL_PASS        cont

In [5]:
# Verifications: schema, test lock, manifest hash, and required output presence.
import json as _json

manifest_hash = sha256_file(MANIFEST_PATH)
assert manifest_hash == EXPECTED_MANIFEST_SHA256, f"manifest hash mismatch: {manifest_hash}"

issues = []
for phase in sorted(FOLDER_BY_PHASE):
    gate = _json.loads((CONF_DATA / phase / "gate_result.json").read_text(encoding="utf-8"))
    for key in ("status", "result_level", "selected_configuration", "selected_metrics",
                "targets", "target_passes", "all_mandatory_targets_passed", "decision",
                "next_step", "manifest_sha256", "input_artifact_hashes", "test_images_accessed"):
        if key not in gate:
            issues.append(f"{phase}: missing gate key {key}")
    if gate.get("test_images_accessed") is not False:
        issues.append(f"{phase}: test_images_accessed not false")
    for req in ("configuration.json", "provenance.json", "expected_vs_actual.csv", "gate_result.json"):
        if not (CONF_DATA / phase / req).is_file():
            issues.append(f"{phase}: missing {req}")

conformance = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "contract_version": "mark1_part2_delivery_contract",
    "phases_covered": len(FOLDER_BY_PHASE),
    "manifest_sha256": manifest_hash,
    "manifest_sha256_matches": manifest_hash == EXPECTED_MANIFEST_SHA256,
    "test_images_accessed": False,
    "schema_issues": issues,
    "all_phases_conform": len(issues) == 0,
    "result_levels": {row.phase: row.result_level for row in summary_df.itertuples(index=False)},
    "summary_file": str(CONF_DATA / "procedure_conformance_summary.csv"),
}
write_json(CONF_DATA / "procedure_conformance_gate.json", conformance)
print("Issues:", issues if issues else "none")
print("all_phases_conform:", conformance["all_phases_conform"])
print("Manifest hash verified:", conformance["manifest_sha256_matches"])


Issues: none
all_phases_conform: True
Manifest hash verified: True


In [6]:
# Conformance dashboard: result levels and gate row pass rates.
order = ["mark_1", "mark_2", "mark_3", "mark_4", "mark_4b", "mark_4c", "mark_4d", "mark_4e", "consolidated"]
level_colors = {"DIAGNOSTIC_COMPLETE": "#1f77b4", "FAILED_GATE": "#d62728",
                "PARTIAL_PASS": "#ff7f0e", "TEMPORARY_CONTINUATION_PASS": "#2ca02c",
                "VALIDATION_FREEZE_PASS": "#17becf", "FINAL_TEST_COMPLETE": "#9467bd"}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df = summary_df.set_index("phase").reindex(order)

left = axes[0]
for i, phase in enumerate(order):
    lvl = df.loc[phase, "result_level"]
    left.barh(i, 1, color=level_colors.get(lvl, "#7f7f7f"))
left.set_yticks(range(len(order)), order)
left.set_xlim(0, 1)
left.set_xticks([])
left.invert_yaxis()
left.set_title("Part 2 result level per phase")
from matplotlib.patches import Patch
handles = [Patch(color=c, label=l) for l, c in level_colors.items() if l in df["result_level"].values]
left.legend(handles=handles, fontsize=8, loc="lower right")

right = axes[1]
right.bar(df.index, df["passed"], color="#2ca02c", label="passed")
right.plot(df.index, df["rows"], "o-", color="#1f77b4", label="gate rows")
right.set_xticks(range(len(order)), order, rotation=30, ha="right")
right.set_ylabel("rows")
right.set_title("Gate rows vs passed (all-mandatory-pass in green)")
right.legend(fontsize=8)
right.grid(alpha=.2)

fig.suptitle("Part 2 procedure conformance across the Evaluation suite")
fig.tight_layout()
fig.savefig(CONF_FIGS / "procedure_conformance_dashboard.png", dpi=170, bbox_inches="tight")
plt.show()
print("Dashboard saved to", CONF_FIGS / "procedure_conformance_dashboard.png")


Dashboard saved to D:\DATA SCIENCE AND ANALYTICS\PROJECTS\Liver\Evaluation\output\12_procedure_conformance\figures\procedure_conformance_dashboard.png


C:\Users\alanm\AppData\Local\Temp\ipykernel_3680\3476776365.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

- Every Evaluation phase now exposes the four Part 2 contract files under
  `Evaluation/output/12_procedure_conformance/data/<phase>/`.
- Result-level mapping: Mark 1–3 `DIAGNOSTIC_COMPLETE`, Mark 4 `FAILED_GATE`,
  Mark 4B `PARTIAL_PASS`, Mark 4C `FAILED_GATE`, Mark 4D `PARTIAL_PASS`,
  Mark 4E `TEMPORARY_CONTINUATION_PASS`, Consolidated `DIAGNOSTIC_COMPLETE`.
- The full machine-readable `gate_result.json` schema (result level, targets, target passes, decision,
  next step, hashes, test lock) is verified for every phase; the manifest hash is asserted.
- Test split remains locked everywhere; no file under `mark 1/` or `mark 1 (part 2)/` is touched.
